In [1]:
from pathlib import Path
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import lognorm
from scipy.integrate import solve_ivp
from scipy.optimize import minimize
from scipy.special import logsumexp


In [2]:
pi = np.pi
kB =  8.617333e-5 #Boltzman constant eV/K

## Data loaders

In [3]:
STANDARD_ROOM_TEMPERATURE_C = 25.0


def parse_dataset_name(filename):
    """
    Parse filenames with format:
        <Temperature>-(B|D)F(-irr*)?

    Examples:
        RT-BF.csv
        RT-BF-irr.csv
        900-BF.csv
        900-DF-irr.csv
    """

    stem = Path(filename).stem

    pattern = r"^(?P<temp>RT|\d+\.?\d*C?)-(?P<mode>[BD]F)(?:-(?P<irr>irr.*))?$"
    match = re.match(pattern, stem, flags=re.IGNORECASE)

    if match is None:
        raise ValueError(
            f"Filename does not match expected format '<Temperature>-(B|D)F(-irr*)?': {filename}"
        )

    temp_raw = match.group("temp").upper()
    mode = match.group("mode").upper()
    irr_tag = match.group("irr")

    if temp_raw == "RT":
        temperature_C = STANDARD_ROOM_TEMPERATURE_C
        temperature_label = "RT"
    else:
        temperature_C = float(temp_raw.replace("C", ""))
        temperature_label = f"{temperature_C:g}C"

    irradiated = irr_tag is not None

    return {
        "temperature_label": temperature_label,
        "temperature_C": temperature_C,
        "mode": mode,
        "irradiated": irradiated,
        "condition": "irradiated" if irradiated else "non-irradiated",
        "irr_tag": irr_tag.lower() if irr_tag is not None else None,
    }
    
def clean_numeric_columns(df):
    cleaned = {}

    for column in df.columns:
        values = pd.to_numeric(df[column], errors="coerce").dropna()

        if values.empty:
            continue

        cleaned[column] = values.to_numpy(dtype=float)

    return cleaned


def load_data(data_dir="Data", pattern="*-[BD]F*.csv"):
    """
    Load loop-size datasets using filename format:
        <Temperature>-(B|D)F(-irr*)?

    Returns
    -------
    datasets : dict
        Dictionary keyed by dataset name.

    loop_data : pandas.DataFrame
        Long-format dataframe useful for plotting and fitting.
    """

    data_dir = Path(data_dir)

    if not data_dir.exists():
        alt_data_dir = Path.cwd().parent / data_dir
        if alt_data_dir.exists():
            data_dir = alt_data_dir
        else:
            raise FileNotFoundError(f"Data directory not found: {data_dir}")

    paths = sorted(data_dir.glob(pattern))

    if len(paths) == 0:
        raise FileNotFoundError(
            f"No files found in {data_dir} matching pattern '{pattern}'"
        )

    datasets = {}
    rows = []

    for path in paths:
        metadata = parse_dataset_name(path.name)

        raw_df = pd.read_csv(path)
        values_by_column = clean_numeric_columns(raw_df)

        if len(values_by_column) == 0:
            print(f"Warning: no numeric data found in {path.name}")
            continue

        dataset_name = path.stem
        all_values = np.concatenate(list(values_by_column.values()))

        datasets[dataset_name] = {
            "path": path,
            "raw": raw_df,
            "values_by_column": values_by_column,
            "all_values": all_values,
            **metadata,
        }

        for column, values in values_by_column.items():
            for value in values:
                rows.append({
                    "dataset": dataset_name,
                    "temperature_label": metadata["temperature_label"],
                    "temperature_C": metadata["temperature_C"],
                    "mode": metadata["mode"],
                    "irradiated": metadata["irradiated"],
                    "condition": metadata["condition"],
                    "irr_tag": metadata["irr_tag"],
                    "column": column,
                    "size": value,
                    "source_file": path.name,
                })

    loop_data = pd.DataFrame(rows)

    return datasets, loop_data
    

## Helpers

In [4]:
def logterminv(R, r0):
    """
    Safe inverse logarithm term.

    Need 8R/r0 > 1 to avoid log <= 0.
    """
    R_min = 1.01 * r0 / 8.0
    R_eff = np.maximum(R, R_min)

    return 1.0 / np.log(8.0 * R_eff / r0)

def j_x_L(Rx, Di, Ci, r0):
    return 2.0 * np.pi**2 * Rx * Di * Ci * logterminv(Rx, r0)


def compute_Rx(Nx, Cx, b, eps_N=1e-300, eps_C=1e-300):
    """
    Safe radius calculation for use inside the ODE.

    Prevents NaN during solve_ivp by clipping Nx and Cx.
    """
    Nx_eff = np.maximum(Nx, eps_N)
    Cx_eff = np.maximum(Cx, eps_C)

    return np.sqrt(Nx_eff / (np.pi * b * Cx_eff))


def diffusion_coeff(D0, Em, T_K):
    return D0 * np.exp(-Em / (kB * T_K))


def coalescence_rate(P0, Ea, T_K):
    return P0 * np.exp(-Ea / (kB * T_K))

In [5]:
def density_to_cm3(value, unit="m^-3"):
    if unit == "m^-3":
        return value * 1e-6
    elif unit == "cm^-3":
        return value
    else:
        raise ValueError("unit must be 'm^-3' or 'cm^-3'")


def get_values(loop_data, temperature_label="RT", mode="DF", irradiated=True):
    subset = loop_data[
        (loop_data["temperature_label"] == temperature_label) &
        (loop_data["mode"] == mode)
    ]

    if "irradiated" in loop_data.columns:
        subset = subset[subset["irradiated"] == irradiated]

    values = subset["size"].to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    values = values[values > 0]

    if len(values) == 0:
        raise ValueError(
            f"No data found for temperature={temperature_label}, mode={mode}, irradiated={irradiated}"
        )

    return values


def lognormal_mean_from_data(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    values = values[values > 0]

    shape, loc, scale = lognorm.fit(values, floc=0)
    return lognorm.mean(shape, loc=loc, scale=scale)


def make_y0_from_RT(
    loop_data,
    BF_density,
    DF_density,
    density_unit="m^-3",
    b=None,
    Omega0=None,   # kept only for compatibility
    Ci0=1e-12,
    Cv0=0.0,
    irradiated=True,
    use_lognormal_mean=True,
):
    """
    Build initial condition from RT data.

    State:
        y0 = [Ci0, Cv0, Nf0, Np0, Cf0, Cp0]
    """

    if b is None:
        raise ValueError("You must pass b.")

    df_values = get_values(loop_data, "RT", "DF", irradiated=irradiated)
    bf_values = get_values(loop_data, "RT", "BF", irradiated=irradiated)

    if use_lognormal_mean:
        Df0_nm = lognormal_mean_from_data(df_values)
        DBF0_nm = lognormal_mean_from_data(bf_values)
    else:
        Df0_nm = np.mean(df_values)
        DBF0_nm = np.mean(bf_values)

    # Convert loop densities to cm^-3
    Ctot0 = density_to_cm3(BF_density, density_unit)
    Cf0 = density_to_cm3(DF_density, density_unit)
    Cp0 = max(Ctot0 - Cf0, 1e-30)

    # Estimate perfect-loop diameter from BF weighted mean:
    # DBF*Ctot = Df*Cf + Dp*Cp
    Dp0_nm = (DBF0_nm * Ctot0 - Df0_nm * Cf0) / Cp0

    if (not np.isfinite(Dp0_nm)) or (Dp0_nm <= 0):
        Dp0_nm = DBF0_nm

    # Convert diameters nm -> radii cm
    Rf0 = 0.5 * Df0_nm * 1e-7
    Rp0 = 0.5 * Dp0_nm * 1e-7

    # Your convention:
    # N = pi*b*R^2*C
    Nf0 = np.pi * b * Rf0**2 * Cf0
    Np0 = np.pi * b * Rp0**2 * Cp0

    y0 = np.array([Ci0, Cv0, Nf0, Np0, Cf0, Cp0], dtype=float)

    print("Initial condition from RT data:")
    print(f"  Df0 = {Df0_nm:.3f} nm")
    print(f"  DBF0 = {DBF0_nm:.3f} nm")
    print(f"  Dp0 estimated = {Dp0_nm:.3f} nm")
    print(f"  Rf0 = {Rf0:.3e} cm = {Rf0 * 1e7:.3f} nm")
    print(f"  Rp0 = {Rp0:.3e} cm = {Rp0 * 1e7:.3f} nm")
    print(f"  Cf0 = {Cf0:.3e} cm^-3")
    print(f"  Cp0 = {Cp0:.3e} cm^-3")
    print(f"  Nf0 = {Nf0:.3e}")
    print(f"  Np0 = {Np0:.3e}")
    print(f"  Ci0 = {Ci0:.3e}")
    print(f"  Cv0 = {Cv0:.3e}")
    print(f"  len(y0) = {len(y0)}")

    return y0

### Plotter

In [6]:
def plot_model_vs_data(
    values_nm,
    mode,
    prediction,
    fit_theta,
    radius_unit_to_nm=1e7,
    title="",
    bins=20,
):
    values_nm = np.asarray(values_nm, dtype=float)
    values_nm = values_nm[np.isfinite(values_nm)]
    values_nm = values_nm[values_nm > 0]

    if len(values_nm) == 0:
        print(f"No valid data for {title}")
        return

    Rf_nm = prediction["Rf"] * radius_unit_to_nm
    Rp_nm = prediction["Rp"] * radius_unit_to_nm

    Df_nm = 2.0 * Rf_nm
    Dp_nm = 2.0 * Rp_nm

    x_max = max(values_nm.max() * 1.2, Df_nm * 1.5, Dp_nm * 1.5)
    x = np.linspace(1e-9, x_max, 500)

    pdf_model = predicted_loop_pdf(
        x_nm=x,
        mode=mode,
        prediction=prediction,
        fit_theta=fit_theta,
        radius_unit_to_nm=radius_unit_to_nm,
    )

    plt.figure(figsize=(7, 5))

    plt.hist(
        values_nm,
        bins=bins,
        density=True,
        alpha=0.65,
        edgecolor="black",
        label="Experimental data",
    )

    plt.plot(
        x,
        pdf_model,
        linewidth=2.5,
        label="RT+ fitted distribution",
    )

    plt.axvline(
        Df_nm,
        linestyle="--",
        linewidth=1.5,
        label=f"Faulted diameter = {Df_nm:.2f} nm",
    )

    if mode == "BF":
        plt.axvline(
            Dp_nm,
            linestyle=":",
            linewidth=1.5,
            label=f"Perfect diameter = {Dp_nm:.2f} nm",
        )

    plt.title(title)
    plt.xlabel("Loop diameter (nm)")
    plt.ylabel("Probability density")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_all_model_vs_data(
    loop_data,
    fit_theta,
    base_params,
    y0_initial,
    fit_temperatures,
    radius_unit_to_nm=1e7,
    t_end_s=3600,
    bins=20,
):
    predictions = {}

    for T_C in fit_temperatures:
        predictions[float(T_C)] = simulate_one_temperature(
            T_C=float(T_C),
            theta=fit_theta,
            base_params=base_params,
            y0=y0_initial,
            t_end_s=t_end_s,
        )

    data_to_plot = loop_data[
        (loop_data["irradiated"] == True) &
        (loop_data["temperature_C"].isin(fit_temperatures))
    ].copy()

    for (T_C, mode), group in data_to_plot.groupby(["temperature_C", "mode"]):
        T_C = float(T_C)
        values_nm = group["size"].to_numpy(dtype=float)

        title = f"{T_C:g} °C - {mode} - irradiated"

        plot_model_vs_data(
            values_nm=values_nm,
            mode=mode,
            prediction=predictions[T_C],
            fit_theta=fit_theta,
            radius_unit_to_nm=radius_unit_to_nm,
            title=title,
            bins=bins,
        )

    return predictions

## Core functions

In [7]:
def ODE(t, y, params):
    #Variables
    Ci, Cv, Nf, Np, Cf, Cp = y

    #Parameters
    Di      = params["Di"]
    Dv      = params["Dv"]
    Rii     = params["Rii"]
    Zii     = params["Zii"]
    Puf     = params["Puf"]
    Pcs     = params["Pcs"]
    b       = params["b"]
    Omega0  = params["Omega0"]
    r0      = params["r0"]
    
    # Optional source terms
    G0i = params.get("G0i", 0.0)
    G0v = params.get("G0v", 0.0)
    Ziv_iK = 48
    Ziv_vK = 48

    geometry_factor = (pi*b)/(3*Omega0)

    # Average loop radii
    Rf = compute_Rx(Nf, Cf, b)
    Rp = compute_Rx(Np, Cp, b)

    # Interstitial fluxes to faulted and perfect loops
    jf = j_x_L(Rf, Di, Ci, r0)
    jp = j_x_L(Rp, Di, Ci, r0)

    kiv = (Omega0/a**2) * (Ziv_iK * Di + Ziv_vK * Dv)

    df = np.zeros(6)

    #Interstitials
    df[0] = G0i  - Rf*Cf*jf - Rp*Cp*jp - 2*Zii*Di*np.power(Ci,2) - kiv * Ci *Cv
    #Vacancies
    df[1] = G0v - kiv * Ci *Cv

    # Total interstitials stored in faulted loops
    df[2] = Rf*Cf*jf - Puf * geometry_factor * np.power(Rf,2) * Cf
    
    # Total interstitials stored in perfect loops
    df[3] = Rp*Cp*jp + Puf * geometry_factor * np.power(Rf,2) * Cf

    # Faulted loop number density
    df[4] = Rii*Di*np.power(Ci,2) - Puf*Cf
    
    # Perfect loop number density
    df[5] = Puf*Cf - Pcs*np.power(Cp,2)
    
    return df

In [8]:
def simulate_one_temperature(T_C, theta, base_params, y0, t_end_s=3600):
    """
    Simulate RT+ model for one temperature.

    State vector:
        y = [Ci, Cv, Nf, Np, Cf, Cp]
    """

    T_C = float(T_C)
    T_K = T_C + 273.15

    Eim = theta["Eim"]
    Evm = theta["Evm"]
    Ea = theta["Ea"]
    P0 = theta["P0"]
    Puf = theta["Puf_by_T"][T_C]

    params = base_params.copy()
    params["Di"] = diffusion_coeff(params["Di0"], Eim, T_K)
    params["Dv"] = diffusion_coeff(params["Dv0"], Evm, T_K)
    params["Pcs"] = coalescence_rate(P0, Ea, T_K)
    params["Puf"] = Puf

    sol = solve_ivp(
        fun=lambda t, y: ODE(t, y, params),
        t_span=(0.0, t_end_s),
        y0=y0,
        method="BDF",
        rtol=1e-6,
        atol=1e-12,
    )

    if not sol.success:
        raise RuntimeError(sol.message)

    y_final = sol.y[:, -1]
    Ci, Cv, Nf, Np, Cf, Cp = y_final

    # Reject nonphysical loop states
    if not np.all(np.isfinite(y_final)):
        raise RuntimeError("Non-finite solution.")

    if Nf <= 0 or Cf <= 0:
        raise RuntimeError("Faulted loop population became nonphysical.")

    if Np <= 0 or Cp <= 0:
        raise RuntimeError("Perfect loop population became nonphysical.")

    Rf = compute_Rx(Nf, Cf, params["b"])
    Rp = compute_Rx(Np, Cp, params["b"])

    if not np.isfinite(Rf) or not np.isfinite(Rp):
        raise RuntimeError("Nonphysical loop radius.")

    return {
        "y_final": y_final,
        "Rf": Rf,
        "Rp": Rp,
        "Cf": Cf,
        "Cp": Cp,
        "Di": params["Di"],
        "Dv": params["Dv"],
        "Pcs": params["Pcs"],
        "Puf": Puf,
    }

In [9]:
def make_y0_from_dataset(
    loop_data,
    temperature_C,
    BF_density,
    DF_density,
    density_unit="m^-3",
    b=None,
    Ci0=1e-12,
    Cv0=0.0,
    irradiated=False,
    use_lognormal_mean=True,
):
    """
    Build initial condition from a selected dataset.

    State:
        y0 = [Ci0, Cv0, Nf0, Np0, Cf0, Cp0]

    Assumptions:
        BF = total loops = faulted + perfect
        DF = faulted loops only
    """

    if b is None:
        raise ValueError("You must pass b.")

    subset_df = loop_data[
        (loop_data["temperature_C"] == float(temperature_C)) &
        (loop_data["mode"] == "DF")
    ]

    subset_bf = loop_data[
        (loop_data["temperature_C"] == float(temperature_C)) &
        (loop_data["mode"] == "BF")
    ]

    if "irradiated" in loop_data.columns:
        subset_df = subset_df[subset_df["irradiated"] == irradiated]
        subset_bf = subset_bf[subset_bf["irradiated"] == irradiated]

    df_values = subset_df["size"].to_numpy(dtype=float)
    bf_values = subset_bf["size"].to_numpy(dtype=float)

    df_values = df_values[np.isfinite(df_values)]
    bf_values = bf_values[np.isfinite(bf_values)]

    df_values = df_values[df_values > 0]
    bf_values = bf_values[bf_values > 0]

    if len(df_values) == 0:
        raise ValueError(f"No DF data found for T={temperature_C}, irradiated={irradiated}")

    if len(bf_values) == 0:
        raise ValueError(f"No BF data found for T={temperature_C}, irradiated={irradiated}")

    if use_lognormal_mean:
        Df0_nm = lognormal_mean_from_data(df_values)
        DBF0_nm = lognormal_mean_from_data(bf_values)
    else:
        Df0_nm = np.mean(df_values)
        DBF0_nm = np.mean(bf_values)

    Ctot0 = density_to_cm3(BF_density, density_unit)
    Cf0 = density_to_cm3(DF_density, density_unit)
    Cp0 = max(Ctot0 - Cf0, 1e-30)

    # Estimate perfect-loop diameter from BF weighted mean
    Dp0_nm = (DBF0_nm * Ctot0 - Df0_nm * Cf0) / Cp0

    if (not np.isfinite(Dp0_nm)) or (Dp0_nm <= 0):
        Dp0_nm = DBF0_nm

    Rf0 = 0.5 * Df0_nm * 1e-7
    Rp0 = 0.5 * Dp0_nm * 1e-7

    Nf0 = np.pi * b * Rf0**2 * Cf0
    Np0 = np.pi * b * Rp0**2 * Cp0

    y0 = np.array([Ci0, Cv0, Nf0, Np0, Cf0, Cp0], dtype=float)

    return y0

## Distributions

In [10]:
def lognormal_shape_from_mean_std(mean, std):
    mean = max(float(mean), 1e-30)
    std = max(float(std), 1e-30)

    return np.sqrt(np.log(1.0 + (std / mean)**2))


def lognormal_logpdf_from_mean_and_k(x, mean, k):
    """
    Lognormal log-PDF where std = k * mean.
    Here k controls the width of the distribution.
    """

    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    x = x[x > 0]

    mean = max(float(mean), 1e-30)
    k = max(float(k), 1e-8)

    std = k * mean
    sigma_logn = lognormal_shape_from_mean_std(mean, std)
    mu_logn = np.log(mean) - 0.5 * sigma_logn**2

    return lognorm.logpdf(x, s=sigma_logn, scale=np.exp(mu_logn))


def predicted_loop_logpdf(values_nm, mode, prediction, fit_theta, radius_unit_to_nm=1e7):
    values_nm = np.asarray(values_nm, dtype=float)
    values_nm = values_nm[np.isfinite(values_nm)]
    values_nm = values_nm[values_nm > 0]

    if len(values_nm) == 0:
        return np.array([])

    Rf_nm = prediction["Rf"] * radius_unit_to_nm
    Rp_nm = prediction["Rp"] * radius_unit_to_nm

    Df_nm = 2.0 * Rf_nm
    Dp_nm = 2.0 * Rp_nm

    Cf = max(float(prediction["Cf"]), 1e-30)
    Cp = max(float(prediction["Cp"]), 1e-30)

    k_f = fit_theta["k_f"]
    k_p = fit_theta["k_p"]

    logpdf_f = lognormal_logpdf_from_mean_and_k(
        x=values_nm,
        mean=Df_nm,
        k=k_f,
    )

    logpdf_p = lognormal_logpdf_from_mean_and_k(
        x=values_nm,
        mean=Dp_nm,
        k=k_p,
    )

    if mode == "DF":
        return logpdf_f

    if mode == "BF":
        wF = Cf / (Cf + Cp)
        wP = Cp / (Cf + Cp)

        return logsumexp(
            np.vstack([
                np.log(wF) + logpdf_f,
                np.log(wP) + logpdf_p,
            ]),
            axis=0,
        )

    raise ValueError(f"Unknown mode: {mode}")


def predicted_loop_pdf(x_nm, mode, prediction, fit_theta, radius_unit_to_nm=1e7):
    return np.exp(
        predicted_loop_logpdf(
            values_nm=x_nm,
            mode=mode,
            prediction=prediction,
            fit_theta=fit_theta,
            radius_unit_to_nm=radius_unit_to_nm,
        )
    )

In [11]:
def unpack_theta(theta_vec, temperatures):
    """
    theta = [
        Eim,
        Evm,
        Ea,
        log(P0),
        log(Puf_T1), ..., log(Puf_Tn),
        log(k_f),
        log(k_p)
    ]
    """

    Eim = theta_vec[0]
    Evm = theta_vec[1]
    Ea = theta_vec[2]
    P0 = np.exp(theta_vec[3])

    Puf_by_T = {}

    # IMPORTANT:
    # Puf starts at index 4, not index 3.
    idx = 4

    for T in temperatures:
        Puf_by_T[float(T)] = np.exp(theta_vec[idx])
        idx += 1

    k_f = np.exp(theta_vec[idx])
    k_p = np.exp(theta_vec[idx + 1])

    return {
        "Eim": Eim,
        "Evm": Evm,
        "Ea": Ea,
        "P0": P0,
        "Puf_by_T": Puf_by_T,
        "k_f": k_f,
        "k_p": k_p,
    }


def build_theta0_and_bounds(temperatures):
    Eim0 = 1.8
    Evm0 = 2.8
    Ea0 = 1.9
    P0_0 = 1e-24

    Puf0_by_T = [1e-5 for _ in temperatures]

    k_f0 = 0.5
    k_p0 = 0.5

    theta0 = np.array(
        [Eim0, Evm0,Ea0, np.log(P0_0)]
        + [np.log(x) for x in Puf0_by_T]
        + [np.log(k_f0), np.log(k_p0)],
        dtype=float,
    )

    bounds = []

    bounds.append((0.1, 6.0))                    # Eim
    bounds.append((0.1, 6.0))                    # Evm
    bounds.append((0.1, 6.0))                    # Ea
    bounds.append((np.log(1e-35), np.log(1e-16))) # P0

    for _ in temperatures:
        bounds.append((np.log(1e-10), np.log(1e-2))) # Puf(T)

    bounds.append((np.log(0.05), np.log(3.0)))   # k_f
    bounds.append((np.log(0.05), np.log(3.0)))   # k_p

    return theta0, bounds


def objective(
    theta_vec,
    loop_data,
    base_params,
    y0_initial,
    fit_temperatures,
    radius_unit_to_nm=1e7,
    t_end_s=3600,
):
    theta = unpack_theta(theta_vec, fit_temperatures)

    total_nll = 0.0
    predictions = {}

    for T_C in fit_temperatures:
        try:
            predictions[float(T_C)] = simulate_one_temperature(
                T_C=float(T_C),
                theta=theta,
                base_params=base_params,
                y0=y0_initial,
                t_end_s=t_end_s,
            )
        except Exception:
            return 1e100

    data_to_fit = loop_data[
        (loop_data["irradiated"] == True) &
        (loop_data["temperature_C"].isin(fit_temperatures))
    ].copy()

    if len(data_to_fit) == 0:
        return 1e100

    for (T_C, mode), group in data_to_fit.groupby(["temperature_C", "mode"]):
        values_nm = group["size"].to_numpy(dtype=float)

        logpdf = predicted_loop_logpdf(
            values_nm=values_nm,
            mode=mode,
            prediction=predictions[float(T_C)],
            fit_theta=theta,
            radius_unit_to_nm=radius_unit_to_nm,
        )

        if len(logpdf) == 0:
            return 1e100

        if not np.all(np.isfinite(logpdf)):
            return 1e100

        total_nll += -np.mean(logpdf)

    if not np.isfinite(total_nll):
        return 1e100

    return total_nll

## Fitting strategy

In [12]:
def lognormal_shape_from_mean_std(mean, std):
    mean = max(float(mean), 1e-30)
    std = max(float(std), 1e-30)

    return np.sqrt(np.log(1.0 + (std / mean)**2))


def lognormal_logpdf_from_mean_and_k(x, mean, k):
    """
    Lognormal log-PDF where std = k * mean.
    Here k controls the width of the distribution.
    """

    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    x = x[x > 0]

    mean = max(float(mean), 1e-30)
    k = max(float(k), 1e-8)

    std = k * mean
    sigma_logn = lognormal_shape_from_mean_std(mean, std)
    mu_logn = np.log(mean) - 0.5 * sigma_logn**2

    return lognorm.logpdf(x, s=sigma_logn, scale=np.exp(mu_logn))


def predicted_loop_logpdf(values_nm, mode, prediction, fit_theta, radius_unit_to_nm=1e7):
    values_nm = np.asarray(values_nm, dtype=float)
    values_nm = values_nm[np.isfinite(values_nm)]
    values_nm = values_nm[values_nm > 0]

    if len(values_nm) == 0:
        return np.array([])

    Rf_nm = prediction["Rf"] * radius_unit_to_nm
    Rp_nm = prediction["Rp"] * radius_unit_to_nm

    Df_nm = 2.0 * Rf_nm
    Dp_nm = 2.0 * Rp_nm

    Cf = max(float(prediction["Cf"]), 1e-30)
    Cp = max(float(prediction["Cp"]), 1e-30)

    k_f = fit_theta["k_f"]
    k_p = fit_theta["k_p"]

    logpdf_f = lognormal_logpdf_from_mean_and_k(
        x=values_nm,
        mean=Df_nm,
        k=k_f,
    )

    logpdf_p = lognormal_logpdf_from_mean_and_k(
        x=values_nm,
        mean=Dp_nm,
        k=k_p,
    )

    if mode == "DF":
        return logpdf_f

    if mode == "BF":
        wF = Cf / (Cf + Cp)
        wP = Cp / (Cf + Cp)

        return logsumexp(
            np.vstack([
                np.log(wF) + logpdf_f,
                np.log(wP) + logpdf_p,
            ]),
            axis=0,
        )

    raise ValueError(f"Unknown mode: {mode}")


def predicted_loop_pdf(x_nm, mode, prediction, fit_theta, radius_unit_to_nm=1e7):
    return np.exp(
        predicted_loop_logpdf(
            values_nm=x_nm,
            mode=mode,
            prediction=prediction,
            fit_theta=fit_theta,
            radius_unit_to_nm=radius_unit_to_nm,
        )
    )

## Implementation

In [13]:
#Load data
datasets, loop_data = load_data(data_dir="Data")

irr_data = loop_data[loop_data["irradiated"]]
non_irr_data = loop_data[~loop_data["irradiated"]]

In [14]:
# -----------------------------
# Constants
# -----------------------------

a = 5.41e-8  # cm
b = (a / 2.0) * np.sqrt(2.0)
Omega0 = (a**3) / 4.0
r0 = 2.0 * a
Zii = 12.0
Rii = (np.sqrt(3.0) / 2.0) * a


# -----------------------------
# Base parameters
# -----------------------------

base_params = {
    "b": b,
    "Omega0": Omega0,
    "r0": r0,
    "Di0": 1e6,
    "Dv0": 1e6,
    "Rii": Rii,
    "Zii": Zii,
}


# -----------------------------
# Random physical initial condition
# -----------------------------

def loguniform(low, high, rng):
    return np.exp(rng.uniform(np.log(low), np.log(high)))


def make_random_y0(b, seed=None):
    rng = np.random.default_rng(seed)

    Ci0 = loguniform(1e-18, 1e-6, rng)
    Cv0 = 0.0

    Cf0 = loguniform(1e14, 1e18, rng)
    Cp0 = loguniform(1e12, 1e18, rng)

    Rf0_nm = loguniform(0.5, 10.0, rng)
    Rp0_nm = loguniform(0.5, 20.0, rng)

    Rf0 = Rf0_nm * 1e-7
    Rp0 = Rp0_nm * 1e-7

    Nf0 = np.pi * b * Rf0**2 * Cf0
    Np0 = np.pi * b * Rp0**2 * Cp0

    y0 = np.array([Ci0, Cv0, Nf0, Np0, Cf0, Cp0], dtype=float)

    print("Random physical y0:")
    print(f"  Ci0 = {Ci0:.3e}")
    print(f"  Cv0 = {Cv0:.3e}")
    print(f"  Rf0 = {Rf0_nm:.3f} nm")
    print(f"  Rp0 = {Rp0_nm:.3f} nm")
    print(f"  Cf0 = {Cf0:.3e} cm^-3")
    print(f"  Cp0 = {Cp0:.3e} cm^-3")
    print(f"  Nf0 = {Nf0:.3e}")
    print(f"  Np0 = {Np0:.3e}")
    print(f"  len(y0) = {len(y0)}")

    return y0


y0_initial = make_random_y0(b=b, seed=10)


# -----------------------------
# Fit temperatures
# -----------------------------

available_temperatures = sorted(
    float(T) for T in loop_data[loop_data["irradiated"]]["temperature_C"].dropna().unique()
)

print("Available irradiated temperatures:", available_temperatures)

FIT_TEMPERATURES = [T for T in available_temperatures if T >= 900.0]
print("Fit temperatures:", FIT_TEMPERATURES)


# -----------------------------
# Fit
# -----------------------------

theta0, bounds = build_theta0_and_bounds(FIT_TEMPERATURES)

result = minimize(
    objective,
    theta0,
    args=(loop_data, base_params, y0_initial, FIT_TEMPERATURES),
    method="L-BFGS-B",
    bounds=bounds,
    options={
        "maxiter": 5000,
        "ftol": 1e-9,
        "gtol": 1e-6,
    },
)

fit_theta = unpack_theta(result.x, FIT_TEMPERATURES)

print("Fit success:", result.success)
print("Message:", result.message)
print("Final objective:", result.fun)
print("Fit parameters:")
print(fit_theta)

Random physical y0:
  Ci0 = 2.965e-07
  Cv0 = 0.000e+00
  Rf0 = 0.782 nm
  Rp0 = 3.315 nm
  Cf0 = 6.772e+14 cm^-3
  Cp0 = 9.347e+16 cm^-3
  Nf0 = 4.977e-07
  Np0 = 1.235e-03
  len(y0) = 6
Available irradiated temperatures: [25.0, 900.0, 1100.0]
Fit temperatures: [900.0, 1100.0]
Fit success: True
Message: CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
Final objective: 1e+100
Fit parameters:
{'Eim': np.float64(1.8), 'Evm': np.float64(2.8), 'Ea': np.float64(1.9), 'P0': np.float64(1.000000000000002e-24), 'Puf_by_T': {900.0: np.float64(9.999999999999997e-06), 1100.0: np.float64(9.999999999999997e-06)}, 'k_f': np.float64(0.5), 'k_p': np.float64(0.5)}


### Plotting

In [15]:
predictions = plot_all_model_vs_data(
    loop_data=loop_data,
    fit_theta=fit_theta,
    base_params=base_params,
    y0_initial=y0_initial,
    fit_temperatures=FIT_TEMPERATURES,
    radius_unit_to_nm=1e7,
    t_end_s=3600,
    bins=20,
)

RuntimeError: Faulted loop population became nonphysical.